# Should Games Go on Sale? Measuring the Impact of Steam Discounts

Project by Jasper Burkey · Impact-Driven Business Decisions, University of Washington, Autumn 2026

---

Four times a year Valve runs a Steam-wide sale, and every game publisher decides whether to discount and how deep. This notebook measures what that decision buys: how much a discount raises demand during the sale, whether the gain lasts or is borrowed from later weeks, and for which games it pays off.

The project follows the course's ***Learn · Decide · Repeat*** loop. It learns from the seasonal sales of 2023–2026, turns the estimates into evidence for a discount decision, and is built so that every new sale can test the results again.

**Contents:** 1. Data · 2. Identification · 3. When the effect happens · 4. How large it is · 5. Robustness · 6. For which games · 7. Conclusions and limitations

In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from auxiliary import data, dml, event_study, plots

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)

## 1. Data

Steam does not publish sales, so the outcome is **review activity**, the only public measure of purchases over time (a buyer must own a game to review it). The data come from public sources and are built by the scripts in [`pipeline/`](pipeline/README.md):

* **Reviews:** every review since January 2023 for each game, from Steam's review API (`IUserReviewsService/GetAppReviews`). For the largest games, exact counts per time window instead of individual reviews.
* **Prices and discounts:** each game's recorded Steam price history from IsThereAnyDeal.
* **Sale dates:** Valve's announced seasonal sales (`pipeline/sale_calendar.csv`).
* **Game characteristics:** release dates from a daily store collector ([`collector/`](collector/README.md)), which has recorded every Steam app since November 2026.

The unit of analysis is **one game in one seasonal sale**. `data/events.parquet` holds one row per game-event: whether the game was discounted, controls measured before the sale, and outcomes. Raw reviews are not included in this repository (they are large and contain hashed reviewer IDs).

In [ ]:
events = data.load_events()
main_df = data.analysis_sample(events)
print(
    f"{len(events):,} game-events: {events['app_id'].nunique():,} games x {events['sale'].nunique()} sale events"
)
print(
    f"Analysis sample: {len(main_df):,} game-events from {main_df['app_id'].nunique():,} games, "
    f"{main_df['discounted'].mean():.0%} discounted"
)
display(data.exclusion_table(events))

Game-events are excluded when the game has too little review history around the sale, its price is unknown, it was released less than 60 days before, it is free, or it was already discounted in the 7 days before the sale (its baseline would be distorted). Most games that join a sale discount deeply:

In [ ]:
display(data.event_summary(events))

## 2. Identification

Valve sets the sale dates; each publisher chooses whether to discount. Comparing discounted with non-discounted games is therefore biased if publishers discount for reasons that also move demand. The design handles this in two ways:

1. **Within-event comparison.** Discounted and non-discounted games are compared *in the same sale event*, so anything that affects every game during the event (front-page traffic, holiday demand) cancels out.
2. **Pre-event controls.** Each estimate adjusts for what is observed before the sale: review rate and trend over the previous 60 days, share of positive reviews, game age, regular price, time since the last discount, and the event itself.

The identifying assumption is **unconfoundedness**: given these pre-event controls, whether a game discounts is unrelated to how its demand would otherwise have moved. The week before each sale serves as a placebo, and the overlap check in section 4 shows where the comparison is well supported.

**Outcomes** are log changes in reviews relative to each game's own baseline (days −21 to −8), measured as 14-day-equivalent totals on a log(1 + x) scale; exp(effect) − 1 is approximately the percentage change.

## 3. When the effect happens

The event study shows, day by day, the difference between discounted and non-discounted games in the same event, relative to their own baselines. It shows the **shape** of the effect: when it starts, how long it lasts, whether it reverses. Single days have few reviews, so this daily scale understates the size; section 4 gives the size.

In [ ]:
curve, n_events = event_study.event_study(events, data.load_daily())
plots.plot_event_study(curve, n_events)
display(event_study.summarize(curve).to_frame("average difference"))

## 4. How large the effect is

Double machine learning estimates the effect of discounting with flexible (gradient-boosted) models for the controls, cross-fitted by game so no game's own data predicts itself. Two estimators are reported: a partially linear model (PLR) and a doubly robust AIPW estimator, for all games (ATE) and for games that discounted (ATT). Standard errors are clustered by game.

In [ ]:
main_table, aipw_fit = dml.main_estimates(main_df)
display(dml.tidy(main_table))

**Timing.** Reviews arrive after purchases, so the effect is also measured before and after the sale. A negative week before suggests buyers wait for announced sales; a negative effect weeks after would suggest the sale only pulled purchases forward. The **net** effect, from a week before to four weeks after, is the best single summary.

In [ ]:
display(dml.tidy(dml.timing_estimates(main_df)))

**Overlap.** Estimates are credible where discounted and non-discounted games with similar histories both exist. Propensity scores near 0 or 1 mark games whose decision is almost fully predictable from their history.

In [ ]:
e = aipw_fit["propensity"]
plots.plot_overlap(e, main_df["discounted"].to_numpy(int))
print(
    f"Share of game-events with propensity in [0.1, 0.9]: {((e >= 0.1) & (e <= 0.9)).mean():.0%}"
)
print(
    f"Comparison group: {(~main_df['discounted']).sum():,} non-discounted game-events "
    f"from {main_df.loc[~main_df['discounted'], 'app_id'].nunique():,} games"
)

## 5. Robustness

In [ ]:
display(dml.tidy(dml.robustness(events, main_df, e)))

## 6. For which games

Effects within three dimensions chosen in advance, from the AIPW scores. The test table asks whether the groups in each dimension differ, with Holm-adjusted p-values for testing three dimensions. Group differences without a small adjusted p-value are not established.

In [ ]:
groups, tests = dml.heterogeneity(main_df, aipw_fit["phi"])
display(dml.tidy(groups))
display(tests.round(3))

## 7. Conclusions and limitations

The tables above give the size of the discount effect during the sale, its net effect once the weeks before and after are counted, and how it varies with discount depth. Read them with four limitations in mind:

* **Reviews are a proxy for purchases.** If discount buyers review at a different rate than full-price buyers, the effect on purchases differs from the effect on reviews.
* **Unconfoundedness is assumed.** The within-event comparison, pre-event controls and placebo week support it, but cannot prove it. Where few comparable games exist (section 4), estimates lean on the model.
* **Scope.** The sample definition (released, paid games with enough reviews) is set in `pipeline/`; results describe that population, not free-to-play games.
* **Six weeks of follow-up.** Effects beyond six weeks after a sale are not measured.

**Next steps.** The daily collector records prices, player counts, top-seller ranks and front-page placement for every sale from Winter 2026 on, which allows an out-of-sample replication with better outcomes. A design based on Steam's 30-day release rule (games released just before versus just after the eligibility cutoff) tests the effect without the unconfoundedness assumption.